# HazeSignal: can fires and wind warn of haze earlier?

**Hypothesis:** more fire hotspots in Sumatra and Kalimantan, when winds carry air from those regions toward Kuala Lumpur, are associated with higher ground-level PM2.5 one or two days later.

This notebook uses September 2023 as a complete, reproducible pilot because the selected OpenAQ monitor has data for that month. September 2019 remains the intended severe-haze case study, but it cannot be tested honestly until daily DOE/APIMS PM2.5 observations are obtained. We keep the intermediate tables visible and treat a messy result as useful evidence, not something to hide.

## 1. Load the small analysis helpers

The notebook uses the same TypeScript functions as the command-line analysis. Run it from the repository's `notebooks` folder with the Deno Jupyter kernel.

In [ ]:
import { parse } from "npm:csv-parse@7.0.2/sync";
import { combineDailyData, dailyWind, initialBearing, KUALA_LUMPUR, linearRegression, scatterSvg, SOURCE_CENTROIDS, windTravelBearing, alignmentScore } from "../src/analysis.ts";
const readCsv = async <T>(path: string): Promise<T[]> => parse(await Deno.readTextFile(path), { columns: true, skip_empty_lines: true, trim: true }) as T[];

## 2. Check the source files

The repository includes the September 2023 CSV files produced by the three fetchers. API keys are never written into those files. Their dates and sources are recorded in `data/README.md`.

In [ ]:
const paths = {
  firms: "../data/firms_2023-09-01_2023-09-30.csv",
  wind: "../data/wind_2023-09-01_2023-09-30.csv",
  pm25: "../data/pm25_2023-09-01_2023-09-30.csv",
};

for (const [name, path] of Object.entries(paths)) {
  try { await Deno.stat(path); console.log(`✓ ${name}: ${path}`); }
  catch { console.log(`✗ ${name}: ${path} is missing`); }
}

## 3. Fire hotspots during the pilot period

The FIRMS file contains one row per VIIRS detection. Counting rows by acquisition date gives our simplest fire-activity measure. This ignores fire intensity for now; later work should use the `frp` column as well.

In [ ]:
const hotspots = await readCsv<Record<string, string>>(paths.firms);
const fireCounts = Object.groupBy(hotspots, row => row.acq_date);
console.table(Object.entries(fireCounts).slice(0, 10).map(([date, rows]) => ({ date, hotspot_count: rows?.length ?? 0 })));

## 4. Wind direction and the source-to-Malaysia bearing

A compass bearing is calculated from each source centre to Kuala Lumpur. If `lat₁, lon₁` is the source and `lat₂, lon₂` is Kuala Lumpur, the initial bearing is:

`atan2(sin(Δlon) cos(lat₂), cos(lat₁) sin(lat₂) − sin(lat₁) cos(lat₂) cos(Δlon))`

Meteorological direction says where wind comes **from**, so smoke travel is `(wind direction + 180°) mod 360°`. The alignment score is `max(0, cos(travel bearing − route bearing))`. This makes the vector calculation visible instead of hiding it in a weather library.

In [ ]:
const routes = Object.entries(SOURCE_CENTROIDS).map(([region, source]) => ({
  region,
  bearing_to_kl: initialBearing(source.latitude, source.longitude, KUALA_LUMPUR.latitude, KUALA_LUMPUR.longitude),
}));
console.table(routes);

const workedExample = { wind_from: 225, smoke_travels_toward: windTravelBearing(225) };
console.log(workedExample, "Sumatra alignment:", alignmentScore(225, routes[0].bearing_to_kl));

In [ ]:
const rawWind = await readCsv<Record<string, string>>(paths.wind);
const wind = dailyWind(rawWind.map(row => ({
  date: row.date,
  wind_speed_kmh: Number(row.wind_speed_kmh),
  wind_direction_degrees: Number(row.wind_direction_degrees),
})));
console.table(wind.slice(0, 10));

## 5. Ground-level PM2.5

This pilot uses daily OpenAQ PM2.5 from sensor 2085316 in Kuala Lumpur. Coverage is mostly high, but 27 September is missing and 28 September has only 29% coverage. We keep that information visible because an incomplete daily average can influence a 30-day regression. OpenAQ's record cannot validate the September 2019 episode because this sensor begins in 2022.

In [ ]:
const rawPm25 = await readCsv<Record<string, string>>(paths.pm25);
const pm25 = rawPm25.map(row => ({ date: row.date, pm25_ug_m3: Number(row.pm25_ug_m3) }));
console.table(rawPm25.map(row => ({
  date: row.date, pm25_ug_m3: Number(row.pm25_ug_m3), coverage_percent: Number(row.coverage_percent),
})));

## 6. Combine each date with later PM2.5

For every wind date, we count hotspots, calculate how well the wind aligns for each hotspot's region, and look up PM2.5 on the next two calendar days. `aligned_hotspot_count` is the sum of the individual alignment scores; a fire contributes 1 when wind points directly toward Kuala Lumpur and 0 when it blows across or away.

In [ ]:
const combined = combineDailyData(hotspots.map(row => ({
  acq_date: row.acq_date,
  region: row.region as "sumatra" | "kalimantan",
})), wind, pm25);
console.table(combined.map(row => ({
  date: row.date, hotspot_count: row.hotspot_count, wind_alignment: row.wind_alignment.toFixed(2),
  aligned_hotspots: row.aligned_hotspot_count.toFixed(1), pm25: row.pm25_ug_m3,
  pm25_next_day: row.pm25_next_day, pm25_in_two_days: row.pm25_in_two_days,
})));

## 7. Simple next-day regression

We fit `next-day PM2.5 = intercept + slope × predictor`. First we use hotspot count alone, then the wind-aligned hotspot count. This comparison matters: the hypothesis says wind should add useful information beyond fires alone. Pearson's `r` describes the direction and strength of the straight-line association. `r²` is the fraction of variation described by one fitted line in this sample; it is not forecast accuracy.

In [ ]:
const rawNextDayPoints = combined.flatMap(row => row.pm25_next_day == null ? [] : [{ x: row.hotspot_count, y: row.pm25_next_day }]);
const alignedNextDayPoints = combined.flatMap(row => row.pm25_next_day == null ? [] : [{ x: row.aligned_hotspot_count, y: row.pm25_next_day }]);
const rawNextDay = linearRegression(rawNextDayPoints);
const alignedNextDay = linearRegression(alignedNextDayPoints);
console.table([
  { predictor: "hotspot count", r: rawNextDay.r, r_squared: rawNextDay.rSquared, observations: rawNextDay.observations },
  { predictor: "wind-aligned hotspots", r: alignedNextDay.r, r_squared: alignedNextDay.rSquared, observations: alignedNextDay.observations },
]);
const svg = scatterSvg(alignedNextDayPoints, alignedNextDay);
await Deno.writeTextFile("../data/regression.svg", svg);
display({ "image/svg+xml": svg }, { raw: true });

## 8. Interpret the result honestly

For September 2023, hotspot count alone has a next-day correlation of `r = 0.637` (`r² = 0.406`), while wind-aligned hotspot count has `r = 0.758` (`r² = 0.574`). The two-day correlations are `0.671` without alignment and `0.710` with alignment. In this sample, adding wind alignment strengthens the association, especially at a one-day lead.

That result is encouraging but fragile. After excluding target PM2.5 days below 75% coverage, the aligned next-day correlation falls to `0.693` (`r² = 0.481`, 27 observations). A few late-month high-fire days also have strong influence, nearby dates are not independent, and the model has no held-out test period. The result is an in-sample association, not proof of causation or a validated forecast.

A fuller model needs several haze and non-haze seasons, fire radiative power, rainfall, humidity, boundary-layer height, wind along the transport route, fire persistence, local-emission controls, and multiple Malaysian monitors. It also needs a held-out period to test whether the relationship predicts new days.

In [ ]:
const rawTwoDayPoints = combined.flatMap(row => row.pm25_in_two_days == null ? [] : [{ x: row.hotspot_count, y: row.pm25_in_two_days }]);
const alignedTwoDayPoints = combined.flatMap(row => row.pm25_in_two_days == null ? [] : [{ x: row.aligned_hotspot_count, y: row.pm25_in_two_days }]);
const rawTwoDay = linearRegression(rawTwoDayPoints);
const alignedTwoDay = linearRegression(alignedTwoDayPoints);
console.table([
  { predictor: "hotspot count", lead: "1 day", r: rawNextDay.r, r_squared: rawNextDay.rSquared, observations: rawNextDay.observations },
  { predictor: "wind-aligned hotspots", lead: "1 day", r: alignedNextDay.r, r_squared: alignedNextDay.rSquared, observations: alignedNextDay.observations },
  { predictor: "hotspot count", lead: "2 days", r: rawTwoDay.r, r_squared: rawTwoDay.rSquared, observations: rawTwoDay.observations },
  { predictor: "wind-aligned hotspots", lead: "2 days", r: alignedTwoDay.r, r_squared: alignedTwoDay.rSquared, observations: alignedTwoDay.observations },
]);

const coverageByDate = new Map(rawPm25.map(row => [row.date, Number(row.coverage_percent)]));
const nextDate = (date: string): string => {
  const value = new Date(`${date}T00:00:00Z`);
  value.setUTCDate(value.getUTCDate() + 1);
  return value.toISOString().slice(0, 10);
};
const reliableNextDayPoints = combined.flatMap(row =>
  row.pm25_next_day == null || (coverageByDate.get(nextDate(row.date)) ?? 0) < 75
    ? []
    : [{ x: row.aligned_hotspot_count, y: row.pm25_next_day }]
);
console.log("Aligned next-day result with at least 75% PM2.5 coverage:");
console.table(linearRegression(reliableNextDayPoints));